# Employee Attrition Prediction — Model Comparison
**ML Assignment | SLIIT | Group Report**

| | |
|---|---|
| **Dataset** | IBM HR Analytics Employee Attrition |
| **Problem** | Binary Classification — Attrition Yes / No |
| **Models compared** | Logistic Regression, Decision Tree, Random Forest, CatBoost |

> Run all 4 individual notebooks first so their CSV files exist in the Performance_Metrics folder, then run this notebook.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#f8f9fa'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.alpha']       = 0.3
plt.rcParams['font.size']        = 11

print("Libraries imported.")

## 2. Load All Model Metrics

In [ ]:
# Paths
METRICS_DIR = r"/Users/sachinthabhashitha/Documents/ML_Assignment/Performance_Metrics"

# Load each model's CSV 
# Decision Tree
dt_df = pd.read_csv(os.path.join(METRICS_DIR, "Decision_Tree_Metrics.csv"))

# Random Forest
rf_df = pd.read_csv(os.path.join(METRICS_DIR, "RandomForest_Metrics.csv"))

# CatBoost
cb_df = pd.read_csv(os.path.join(METRICS_DIR, "CatBoost_Metrics.csv"))


# Replace the values (logistic regression)
lr_df = pd.DataFrame([{
    "Model"         : "Logistic Regression",
    "Algorithm"     : "Logistic Regression",
    "Accuracy (%)"  : None,
    "F1 Score (%)"  : None,
    "Precision (%)" : None,
    "Recall (%)"    : None,
    "ROC-AUC (%)"   : None,
}])

print("Files loaded:")
print(f"  Decision Tree    : {os.path.join(METRICS_DIR, 'Decision_Tree_Metrics.csv')}")
print(f"  Random Forest    : {os.path.join(METRICS_DIR, 'RandomForest_Metrics.csv')}")
print(f"  CatBoost         : {os.path.join(METRICS_DIR, 'CatBoost_Metrics.csv')}")
print(f"  Logistic Reg.    : placeholder (update when available)")

## 3. Build Unified Comparison Table

In [ ]:
# Normalise column names across all CSVs 
# Each member may have used slightly different column names.
# We extract each metric explicitly by its known column name per CSV.

def safe_get(df, col, default=None):
    """Safely get first value of a column, return default if missing."""
    for c in df.columns:
        if c.strip().lower().replace(' ','').replace('_','') == col.lower().replace(' ','').replace('_',''):
            val = df[c].iloc[0]
            return round(float(val), 2) if pd.notna(val) else default
    return default

results = pd.DataFrame([
    {
        "Model"        : "Logistic Regression",
        "Member"       : "Member 1",
        "Notebook"     : "LogisticRegression.ipynb",
        "Accuracy"     : safe_get(lr_df, "Accuracy (%)"),
        "F1 Score"     : safe_get(lr_df, "F1 Score (%)"),
        "Precision"    : safe_get(lr_df, "Precision (%)"),
        "Recall"       : safe_get(lr_df, "Recall (%)"),
        "ROC-AUC"      : safe_get(lr_df, "ROC-AUC (%)"),
    },
    {
        "Model"        : "Decision Tree",
        "Member"       : "Member 3",
        "Notebook"     : "Decision_Tree.ipynb",
        "Accuracy"     : safe_get(dt_df, "Accuracy") if safe_get(dt_df, "Accuracy") and safe_get(dt_df, "Accuracy") < 2 else safe_get(dt_df, "Accuracy"),
        "F1 Score"     : safe_get(dt_df, "F1-Score") or safe_get(dt_df, "F1 Score") or safe_get(dt_df, "F1Score"),
        "Precision"    : safe_get(dt_df, "Precision"),
        "Recall"       : safe_get(dt_df, "Recall"),
        "ROC-AUC"      : safe_get(dt_df, "AUC Score") or safe_get(dt_df, "ROC-AUC") or safe_get(dt_df, "ROCAUC"),
    },
    {
        "Model"        : "Random Forest",
        "Member"       : "Member 2",
        "Notebook"     : "random_forest_attrition.ipynb",
        "Accuracy"     : safe_get(rf_df, "Accuracy") or safe_get(rf_df, "Accuracy (%)"),
        "F1 Score"     : safe_get(rf_df, "F1 Score") or safe_get(rf_df, "F1Score") or safe_get(rf_df, "F1-Score"),
        "Precision"    : safe_get(rf_df, "Precision"),
        "Recall"       : safe_get(rf_df, "Recall"),
        "ROC-AUC"      : safe_get(rf_df, "ROC-AUC") or safe_get(rf_df, "AUC-ROC Score") or safe_get(rf_df, "AUC Score"),
    },
    {
        "Model"        : "CatBoost",
        "Member"       : "Member 4",
        "Notebook"     : "Member4_CatBoost_Final.ipynb",
        "Accuracy"     : safe_get(cb_df, "Accuracy (%)"),
        "F1 Score"     : safe_get(cb_df, "F1 Score (%)"),
        "Precision"    : safe_get(cb_df, "Precision (%)"),
        "Recall"       : safe_get(cb_df, "Recall (%)"),
        "ROC-AUC"      : safe_get(cb_df, "ROC-AUC (%)"),
    },
])

# ── Normalise: if values look like decimals (0.83) convert to percent (83.0) ─
for col in ["Accuracy", "F1 Score", "Precision", "Recall", "ROC-AUC"]:
    results[col] = results[col].apply(
        lambda x: round(x * 100, 2) if pd.notna(x) and x <= 2 else x
    )

print("=" * 75)
print("                  FULL MODEL COMPARISON TABLE")
print("=" * 75)
display_cols = ["Model", "Member", "Accuracy", "F1 Score", "Precision", "Recall", "ROC-AUC"]
print(results[display_cols].to_string(index=False))
print("=" * 75)
print("Note: None = Logistic Regression metrics not yet available")

## 4. Comparison Charts

In [ ]:
# All Metrics Side-by-Side Bar Chart 
metrics    = ["Accuracy", "F1 Score", "Precision", "Recall", "ROC-AUC"]
models     = results["Model"].tolist()
colors     = ["#5C6BC0", "#26A69A", "#EF5350", "#FFA726"]
x          = np.arange(len(metrics))
bar_width  = 0.18

fig, ax = plt.subplots(figsize=(13, 6))
fig.suptitle("Employee Attrition — All Models Performance Comparison",
             fontsize=14, fontweight='bold', y=1.01)

for i, (model, color) in enumerate(zip(models, colors)):
    vals = [results.loc[results["Model"] == model, m].values[0] for m in metrics]
    offset = (i - len(models) / 2 + 0.5) * bar_width
    bars = ax.bar(x + offset, vals, bar_width,
                  label=model, color=color, alpha=0.88, edgecolor='white')
    for bar, val in zip(bars, vals):
        if pd.notna(val):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.5,
                    f"{val:.1f}%",
                    ha='center', va='bottom', fontsize=7.5, fontweight='500')

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 105)
ax.legend(loc='upper right', fontsize=10)
ax.set_xlabel("Metric")
plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, "comparison_all_metrics.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: comparison_all_metrics.png")

In [ ]:
# Radar / Spider Chart
# Shows each model's strengths and weaknesses across all metrics at a glance

from matplotlib.patches import FancyArrowPatch

categories = ["Accuracy", "F1 Score", "Precision", "Recall", "ROC-AUC"]
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.suptitle("Model Comparison — Radar Chart\n(all metrics scaled 0–100%)",
             fontsize=13, fontweight='bold')

colors = ["#5C6BC0", "#26A69A", "#EF5350", "#FFA726"]

for i, (_, row) in enumerate(results.iterrows()):
    vals = [row[m] if pd.notna(row[m]) else 0 for m in categories]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, color=colors[i], label=row['Model'])
    ax.fill(angles, vals, alpha=0.07, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 100)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(["20%", "40%", "60%", "80%", "100%"], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, "comparison_radar.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: comparison_radar.png")

In [ ]:
# Accuracy vs F1 Scatter with ROC-AUC as bubble size 
# Each model is a bubble. Position = Accuracy vs F1. Size = ROC-AUC.
# Best model is top-right with the largest bubble.

fig, ax = plt.subplots(figsize=(9, 6))
fig.suptitle("Accuracy vs F1 Score\n(bubble size = ROC-AUC)",
             fontsize=13, fontweight='bold')

colors = ["#5C6BC0", "#26A69A", "#EF5350", "#FFA726"]

for i, (_, row) in enumerate(results.iterrows()):
    acc = row["Accuracy"]
    f1  = row["F1 Score"]
    auc = row["ROC-AUC"]
    if pd.isna(acc) or pd.isna(f1):
        continue
    size = (auc / 100 * 3000) if pd.notna(auc) else 500
    ax.scatter(acc, f1, s=size, color=colors[i], alpha=0.75,
               edgecolors='white', linewidth=1.5, zorder=3)
    ax.annotate(row["Model"],
                xy=(acc, f1),
                xytext=(8, 8),
                textcoords='offset points',
                fontsize=10, fontweight='500',
                color=colors[i])

ax.set_xlabel("Accuracy (%)", fontsize=11)
ax.set_ylabel("F1 Score (%)", fontsize=11)
ax.set_xlim(60, 95)
ax.set_ylim(0, 70)

# Quadrant lines at midpoints
ax.axvline(x=80, color='gray', linestyle='--', alpha=0.4, lw=1)
ax.axhline(y=40, color='gray', linestyle='--', alpha=0.4, lw=1)
ax.text(80.5, 1, "High Accuracy", fontsize=8, color='gray')
ax.text(61,   41, "High F1",       fontsize=8, color='gray')

# Legend for bubble size
for auc_val, label in [(70, "AUC 70%"), (80, "AUC 80%")]:
    ax.scatter([], [], s=auc_val/100*3000, color='gray', alpha=0.4, label=label)
ax.legend(title="Bubble size", loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, "comparison_scatter.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: comparison_scatter.png")

## 5. Best Model Per Metric

In [ ]:
# Best model per metric
print("=" * 50)
print("        BEST MODEL PER METRIC")
print("=" * 50)

for metric in ["Accuracy", "F1 Score", "Precision", "Recall", "ROC-AUC"]:
    valid = results.dropna(subset=[metric])
    if valid.empty:
        print(f"  {metric:<12}: No data available")
        continue
    best_row = valid.loc[valid[metric].idxmax()]
    print(f"  {metric:<12}: {best_row['Model']:<22} ({best_row[metric]:.2f}%)")

print("=" * 50)

# Overall winner by average rank
print()
print("Overall ranking by average score across all metrics:")
metrics_cols = ["Accuracy", "F1 Score", "Precision", "Recall", "ROC-AUC"]
results["Average Score"] = results[metrics_cols].mean(axis=1)
ranked = results[["Model", "Member", "Average Score"]].dropna().sort_values(
    "Average Score", ascending=False
).reset_index(drop=True)
ranked.index += 1
print(ranked.to_string())

## 6. Export Combined Metrics Table

In [ ]:
# Save combined CSV
combined_path = os.path.join(METRICS_DIR, "All_Models_Comparison.csv")
results.to_csv(combined_path, index=False)

print(f"Combined metrics saved to:")
print(f"  {combined_path}")
print()
print("Files saved to Performance_Metrics folder:")
print("  comparison_all_metrics.png  — bar chart (all 5 metrics)")
print("  comparison_radar.png        — radar/spider chart")
print("  comparison_scatter.png      — accuracy vs F1 bubble chart")
print("  All_Models_Comparison.csv   — combined table")

## 7. Final Discussion

Use this section in your report to discuss the comparison between models.

### Summary of findings

Four supervised learning algorithms were applied to the IBM HR Analytics Employee Attrition dataset (1,470 rows, 30 features after preprocessing). All models addressed the class imbalance (83% No, 17% Yes) using class weighting.

### Algorithm comparison

| Algorithm | Strength | Weakness |
|---|---|---|
| Logistic Regression | Simple, interpretable, fast | Lower accuracy on non-linear data |
| Decision Tree | Visualisable, no scaling needed | Prone to overfitting without pruning |
| Random Forest | High accuracy, robust | Low recall on minority class |
| CatBoost | Best F1 & ROC-AUC, native categorical handling | Slightly more complex to explain |

### Which model is best?

The best model depends on the business objective:
- If **minimising missed attrition cases** (high recall) is the priority → Logistic Regression or Decision Tree
- If **overall accuracy** is the priority → Random Forest or CatBoost
- If **balancing precision and recall** (F1) is the priority → CatBoost
- If **ranking employees by risk** (ROC-AUC) is the priority → CatBoost